# Hồi quy tuyến tính đơn: ảnh hưởng của các số đo cơ thể đến phần trăm mỡ cơ thể

**Môn:** Suy luận thống kê (MI3031) – Chương 4, mục 4.1: Hồi quy tuyến tính đơn
**Nhóm / Sinh viên:** *(điền tên, MSSV)*
**Dữ liệu:** Body Fat (252 nam giới)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm

pd.set_option("display.precision", 3)
plt.rcParams["figure.dpi"] = 110

## 1. Giới thiệu dữ liệu

### 1.1 Nguồn dữ liệu

Bộ dữ liệu **Body Fat** gồm số đo cơ thể của **252 nam giới** (tuổi từ 22 đến 81). Phần trăm mỡ cơ thể được ước lượng từ mật độ cơ thể đo bằng phương pháp cân dưới nước. Dữ liệu do Penrose, Nelson và Fisher (1985) thu thập và được Johnson (1996) giới thiệu trong *Journal of Statistics Education*, 4(1). Báo cáo dùng bản Kaggle của fedesoriano.

- Mô tả biến bản gốc: <https://jse.amstat.org/datasets/fat.txt>
- Bản Kaggle: <https://www.kaggle.com/datasets/fedesoriano/body-fat-prediction-dataset>
- Bản OpenML (id 560): <https://api.openml.org/d/560>; StatLib: <https://lib.stat.cmu.edu/datasets/bodyfat>

### 1.2 Các biến sử dụng

Trong 14 biến của bộ dữ liệu, báo cáo chỉ dùng 5 biến sau (đều là biến số liên tục, chưa dùng biến giả):

| Biến | Ý nghĩa | Đơn vị | Vai trò |
|---|---|---|---|
| `BodyFat` | Phần trăm mỡ cơ thể (công thức Siri) | % | **Y** |
| `Abdomen` | Vòng bụng | cm | x |
| `Wrist` | Vòng cổ tay | cm | x |
| `Age` | Tuổi | năm | x |
| `Height` | Chiều cao | inch | x (biến đối chứng) |

### 1.3 Lý do chọn từng biến x

- **`Abdomen`**: mỡ ở nam giới tích tụ nhiều ở vùng bụng, nên vòng bụng lớn thường đi kèm tỉ lệ mỡ cao.
- **`Age`**: tỉ lệ mỡ cơ thể có xu hướng tăng theo tuổi.
- **`Wrist`**: vòng cổ tay phản ánh kích thước khung xương; người có khung xương lớn thường nặng hơn nên có thể có tỉ lệ mỡ khác. Mối liên hệ này là gián tiếp.
- **`Height`**: chiều cao không có lý do rõ ràng ảnh hưởng đến tỉ lệ mỡ, được đưa vào làm **biến đối chứng** để xem một biến ít liên quan sẽ cho kết quả ra sao.

### 1.4 Các biến không dùng

- `Density`: `BodyFat` được tính trực tiếp từ biến này (`BodyFat = 495/Density − 450`), nên hồi quy theo `Density` là hiển nhiên và không có ý nghĩa phân tích.
- `Weight`, `Chest`, `Hip`, `Thigh`, ...: đều là các số đo kích thước thân người, cùng phản ánh độ "to" của cơ thể như `Abdomen`. Để báo cáo gọn, chỉ chọn 4 biến đại diện cho 4 khía cạnh khác nhau (vòng bụng, khung xương, tuổi, chiều cao). Đây là lựa chọn của báo cáo, không có nghĩa các biến còn lại không liên quan đến `BodyFat`.

### 1.5 Xử lý dữ liệu

Dữ liệu không thiếu và không trùng dòng. Chỉ **một ô** được sửa: quan sát số 42 có `Height` = 29,5 inch (khoảng 75 cm), rõ ràng là lỗi gõ nhầm vì người này nặng 205 lb và vòng bụng 104,3 cm; được sửa thành **69,5 inch** (Johnson, 1996 cũng đề xuất giá trị này). Các điểm đáng ngờ khác được **giữ nguyên**, không loại bỏ: case 182 (`BodyFat` = 0), case 48 và 76 (`BodyFat` lệch khỏi giá trị tính từ `Density`), case 39 (vòng bụng 148,1 cm, lớn hơn hẳn các quan sát còn lại).

### 1.6 Mục tiêu phân tích

Xét ảnh hưởng của **từng** số đo cơ thể (`Abdomen`, `Wrist`, `Age`, `Height`) đến `BodyFat` bằng hồi quy tuyến tính đơn: ước lượng hệ số, kiểm định ý nghĩa thống kê của hệ số góc và so sánh R² giữa các biến.

In [ ]:
# Đọc dữ liệu (file đã sửa case 42) và chọn các cột dùng trong báo cáo
df_full = pd.read_csv("bodyfat_cleaned.csv")
df = df_full[["BodyFat", "Abdomen", "Wrist", "Age", "Height"]].copy()

print("Kích thước toàn bộ dữ liệu:", df_full.shape)
print("Kích thước dữ liệu dùng để hồi quy:", df.shape)

# Kiểm tra ô đã sửa: case 42 là dòng chỉ số 41 (đếm từ 0)
print("\nCase 42 sau khi sửa:")
print(df_full.loc[41, ["Weight", "Height", "Abdomen", "BodyFat"]].to_string())

print("\n5 dòng đầu:")
display(df.head())

print("Thống kê mô tả:")
display(df.describe().round(2))

print("Số giá trị thiếu theo cột:")
print(df.isna().sum().to_string())

**Nhận xét.** Dữ liệu dùng để hồi quy gồm **n = 252** quan sát và 5 biến, không có giá trị thiếu. Chiều cao sau khi sửa case 42 nằm trong khoảng hợp lý (nhỏ nhất khoảng 64 inch). Các biến có đơn vị khác nhau (cm, inch, năm, %), vì vậy hệ số góc ở mục 3 sẽ được diễn giải theo đúng đơn vị của từng biến x và không so sánh trực tiếp độ lớn của các hệ số với nhau; việc so sánh giữa các biến dựa trên R².

## 2. Khảo sát sơ bộ

Trước khi ước lượng mô hình, ta vẽ biểu đồ phân tán của `BodyFat` theo từng biến x để xem: quan hệ có gần dạng đường thẳng hay không (điều kiện để mô hình tuyến tính đơn hợp lý), chiều của quan hệ (đi lên hay đi xuống), và mức độ phân tán của các điểm quanh đường thẳng. Đường màu đỏ là đường hồi quy bình phương nhỏ nhất, chỉ vẽ để minh họa; cách tính và kết luận nằm ở mục 3.

In [ ]:
xs = ["Abdomen", "Wrist", "Age", "Height"]
don_vi = {"Abdomen": "cm", "Wrist": "cm", "Age": "năm", "Height": "inch"}

# Hai điểm đáng chú ý (chỉ số đếm từ 0): case 39 (Abdomen = 148,1) và case 182 (BodyFat = 0)
dac_biet = [38, 181]

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for ax, x in zip(axes.ravel(), xs):
    ax.scatter(df[x], df["BodyFat"], s=14, alpha=0.6, edgecolor="none")
    b1, b0 = np.polyfit(df[x], df["BodyFat"], 1)          # đường thẳng bình phương nhỏ nhất (chỉ để minh họa)
    g = np.linspace(df[x].min(), df[x].max(), 100)
    ax.plot(g, b0 + b1 * g, color="crimson", lw=1.5)
    for i in dac_biet:                                      # khoanh tròn hai điểm đáng chú ý
        ax.scatter(df.loc[i, x], df.loc[i, "BodyFat"], s=70, facecolors="none", edgecolors="black", lw=1.2)
    ax.set_xlabel(f"{x} ({don_vi[x]})")
    ax.set_ylabel("BodyFat (%)")
    ax.set_title(f"BodyFat theo {x}")
fig.suptitle("Biểu đồ phân tán (vòng tròn đen: case 39 và case 182)", y=1.0)
plt.tight_layout()
plt.show()

**Nhận xét về dạng và chiều của quan hệ** (theo biểu đồ phân tán):

- **`Abdomen`**: các điểm đi lên rõ rệt và tập trung quanh một đường thẳng, nên quan hệ **thuận** và **gần tuyến tính**. Độ phân tán của `BodyFat` quanh đường thẳng khá đều ở các mức vòng bụng khác nhau, chưa thấy dạng phễu hay dạng cong. Hai điểm nằm tách khỏi đám mây là case 39 (vòng bụng 148,1 cm nhưng `BodyFat` chỉ 35,2%, thấp hơn nhiều so với đường thẳng) và case 182 (`BodyFat` = 0).
- **`Wrist`**: xu hướng **đi lên** nhưng các điểm phân tán rất rộng quanh đường thẳng, nên quan hệ thuận ở mức yếu.
- **`Age`**: xu hướng **đi lên** nhẹ, đám mây điểm cũng rộng.
- **`Height`**: đường thẳng gần như nằm **ngang**, không thấy xu hướng tăng hay giảm.

**Chọn biến để hồi quy.** Giữ cả bốn biến `Abdomen`, `Wrist`, `Age`, `Height` cho mục 3. Ba biến đầu có xu hướng đi lên có thể quan sát được, còn `Height` được giữ có chủ ý làm trường hợp đối chứng không có quan hệ. Các nhận định ở đây chỉ dựa trên hình vẽ; mức độ và ý nghĩa thống kê của từng quan hệ được đo bằng kiểm định, khoảng tin cậy và $R^2$ ở mục 3.

## 3. Hồi quy tuyến tính đơn cho từng biến x

Với mỗi biến $x$ ta xét mô hình

$$Y_i = \beta_0 + \beta_1 x_i + \varepsilon_i,\qquad \varepsilon_i \ \text{độc lập},\ \varepsilon_i \sim N(0,\sigma^2),\quad i=1,\dots,n,$$

với $Y$ là `BodyFat` (%) và $n = 252$. Các hệ số được ước lượng bằng phương pháp bình phương nhỏ nhất. Các kết quả kiểm định và khoảng tin cậy dưới đây dựa trên giả thiết sai số độc lập, có phương sai không đổi và xấp xỉ chuẩn của mô hình.

Với mỗi biến, ta kiểm định ý nghĩa của hồi quy ở mức ý nghĩa $\alpha = 5\%$ theo 5 bước của bài giảng:

1. Cặp giả thuyết $H_0: \beta_1 = 0$ (không có quan hệ tuyến tính giữa $x$ và $Y$) với $H_1: \beta_1 \ne 0$.
2. Tiêu chuẩn kiểm định $T_0 = \dfrac{\hat\beta_1}{S/\sqrt{S_{xx}}}$; nếu $H_0$ đúng thì $T_0 \sim t(n-2) = t(250)$.
3. Tính giá trị quan sát $t_0 = \hat\beta_1/\mathrm{se}(\hat\beta_1)$.
4. Miền bác bỏ $W_\alpha = (-\infty;\,-t_{0{,}025;250}) \cup (t_{0{,}025;250};\,+\infty)$, với $t_{0{,}025;250} \approx 1{,}9695$. Vì cả bốn mô hình đều có $n = 252$ nên miền bác bỏ giống nhau: $W_\alpha = (-\infty;\,-1{,}9695) \cup (1{,}9695;\,+\infty)$.
5. Nếu $t_0 \in W_\alpha$ thì bác bỏ $H_0$; nếu $t_0 \notin W_\alpha$ thì chưa có cơ sở để bác bỏ $H_0$.

Hệ số chặn $\hat\beta_0$ ứng với $x = 0$ (vòng bụng 0 cm, cổ tay 0 cm, tuổi 0, chiều cao 0 inch) không có ý nghĩa thực tế, nên không diễn giải mà chỉ dùng để viết phương trình.

Biến đầu tiên, `Abdomen`, được tính **bằng công thức** từ các tổng bình phương rồi đối chiếu với `statsmodels`; ba biến còn lại dùng thẳng `statsmodels`. Bảng của `statsmodels` có thêm cột p-value; kết luận ở đây dựa vào miền bác bỏ, p-value chỉ để đối chiếu.

Với `Abdomen` (biến chính), ta làm thêm: kiểm định bằng phân tích phương sai (ANOVA, tiêu chuẩn $F_0$) ở 3.1.1, và khoảng tin cậy cho $\beta_0$, $\beta_1$ ở 3.1.2.

In [ ]:
# --- Hồi quy BodyFat theo Abdomen: tính tay từ công thức ---
x = df["Abdomen"].to_numpy()
y = df["BodyFat"].to_numpy()
n = len(x)

# Bước tính các tổng bình phương
Sxx = np.sum(x**2) - np.sum(x)**2 / n
Sxy = np.sum(x * y) - np.sum(x) * np.sum(y) / n
SST = np.sum(y**2) - np.sum(y)**2 / n

# Ước lượng hệ số (phương pháp bình phương nhỏ nhất)
b1 = Sxy / Sxx
b0 = y.mean() - b1 * x.mean()

# Phân tích phương sai và ước lượng độ lệch chuẩn của sai số
SSR = b1 * Sxy
SSE = SST - SSR
s2 = SSE / (n - 2)
s = np.sqrt(s2)

# Kiểm định H0: beta1 = 0 với H1: beta1 != 0, alpha = 5%
se_b1 = s / np.sqrt(Sxx)
t0 = b1 / se_b1                              # bước 3: giá trị quan sát của T0
t_crit = stats.t.ppf(0.975, n - 2)           # bước 4: t(0,025; n-2) để lập miền bác bỏ
trong_mien_bac_bo = abs(t0) > t_crit         # bước 5: t0 có thuộc W hay không

R2 = SSR / SST
ci_low, ci_high = b1 - t_crit * se_b1, b1 + t_crit * se_b1

print(f"n = {n}")
print(f"Sxx = {Sxx:.2f}   Sxy = {Sxy:.2f}   SST = {SST:.2f}")
print(f"b1 = {b1:.4f}   b0 = {b0:.4f}")
print(f"SSR = {SSR:.2f}   SSE = {SSE:.2f}   s = {s:.4f}")
print(f"se(b1) = {se_b1:.5f}")
print()
print("Kiểm định H0: beta1 = 0 (alpha = 5%)")
print(f"  Giá trị quan sát t0 = {t0:.3f}")
print(f"  Miền bác bỏ W = (-inf; {-t_crit:.4f}) U ({t_crit:.4f}; +inf)")
print(f"  t0 thuộc W? {trong_mien_bac_bo}  ->", "bác bỏ H0" if trong_mien_bac_bo else "chưa có cơ sở để bác bỏ H0")
print()
print(f"R^2 = {R2:.4f}")
print(f"CI 95% cho beta1: ({ci_low:.4f}; {ci_high:.4f})")

# Đối chiếu với statsmodels
res_check = sm.OLS(y, sm.add_constant(x)).fit()
khop = np.allclose([b0, b1, se_b1, t0, R2],
                   [res_check.params[0], res_check.params[1], res_check.bse[1],
                    res_check.tvalues[1], res_check.rsquared])
print("\nKhớp với statsmodels:", khop)

### 3.1 `Abdomen`

**Phương trình hồi quy:** $\widehat{\text{BodyFat}} = -39{,}28 + 0{,}6313\,\text{Abdomen}$. Khi vòng bụng tăng thêm 1 cm thì phần trăm mỡ cơ thể trung bình tăng thêm khoảng 0,63 điểm phần trăm. Độ lệch chuẩn ước lượng của sai số là $s \approx 4{,}88$ (điểm phần trăm).

**Kiểm định ý nghĩa của hồi quy** ($\alpha = 5\%$):

1. Cặp giả thuyết: $H_0: \beta_1 = 0$, $H_1: \beta_1 \ne 0$.
2. Tiêu chuẩn kiểm định $T_0 = \hat\beta_1/(S/\sqrt{S_{xx}})$; nếu $H_0$ đúng thì $T_0 \sim t(250)$.
3. Giá trị quan sát: $t_0 = \dfrac{0{,}6313}{0{,}02855} = 22{,}11$.
4. Miền bác bỏ: $W_\alpha = (-\infty;\,-1{,}9695) \cup (1{,}9695;\,+\infty)$.
5. Vì $t_0 = 22{,}11 \in W_\alpha$ nên **bác bỏ $H_0$**: vòng bụng có ảnh hưởng tuyến tính có ý nghĩa thống kê đến `BodyFat`.

**Khoảng tin cậy 95% cho $\beta_1$:** $(0{,}5751;\ 0{,}6875)$, không chứa 0, phù hợp với kết luận bác bỏ $H_0$.

**Hệ số xác định:** $R^2 = 0{,}6617$, tức vòng bụng giải thích được khoảng **66,2%** biến thiên của `BodyFat`.

In [ ]:
# --- Kiểm định bằng phân tích phương sai (ANOVA) cho Abdomen: dùng lại SSR, SSE, SST ở trên ---
MSR = SSR / 1
MSE = SSE / (n - 2)
f0 = MSR / MSE                                 # bước 3: giá trị quan sát của F0
f_crit = stats.f.ppf(0.95, 1, n - 2)           # bước 4: f(0,05; 1; n-2) để lập miền bác bỏ

anova = pd.DataFrame({
    "Tổng bình phương": [SSR, SSE, SST],
    "Bậc tự do": [1, n - 2, n - 1],
    "Bình phương trung bình": [MSR, MSE, np.nan],
    "f0": [f0, np.nan, np.nan]},
    index=["Hồi quy", "Sai số", "Tổng"])
display(anova.round(3))

print(f"Miền bác bỏ W = ({f_crit:.4f}; +inf)")
print(f"f0 thuộc W? {f0 > f_crit}  ->", "bác bỏ H0" if f0 > f_crit else "chưa có cơ sở để bác bỏ H0")
print(f"Đối chiếu: t0^2 = {t0**2:.2f}   Khớp statsmodels: {np.isclose(f0, res_check.fvalue)}")

#### 3.1.1 Kiểm định bằng phân tích phương sai (ANOVA) cho `Abdomen`

Từ $SST = SSR + SSE$ ta lập bảng ANOVA và kiểm định cùng cặp giả thuyết ($\alpha = 5\%$):

| Nguồn | Tổng bình phương | Bậc tự do | Bình phương trung bình | $f_0$ |
|---|---|---|---|---|
| Hồi quy | $SSR = 11631{,}53$ | 1 | $MSR = 11631{,}53$ | $488{,}93$ |
| Sai số | $SSE = 5947{,}46$ | 250 | $MSE = 23{,}79$ | |
| Tổng | $SST = 17578{,}99$ | 251 | | |

1. Cặp giả thuyết: $H_0: \beta_1 = 0$, $H_1: \beta_1 \ne 0$.
2. Tiêu chuẩn kiểm định $F_0 = \dfrac{SSR/1}{SSE/(n-2)} = \dfrac{MSR}{MSE}$; nếu $H_0$ đúng thì $F_0 \sim F(1,\,250)$.
3. Giá trị quan sát: $f_0 = 11631{,}53/23{,}79 = 488{,}93$.
4. Miền bác bỏ: $W_\alpha = (f_{0{,}05;1;250};\,+\infty) = (3{,}8789;\,+\infty)$.
5. Vì $f_0 = 488{,}93 \in W_\alpha$ nên **bác bỏ $H_0$**, cùng kết luận với kiểm định $T_0$ ở trên. Số liệu cũng cho thấy $f_0 = t_0^2$ ($22{,}11^2 \approx 488{,}9$): trong hồi quy đơn, hai cách kiểm định này luôn cho cùng kết luận.

In [ ]:
# --- Khoảng tin cậy 95% cho beta0 và beta1 (Abdomen) ---
x_bar = x.mean()
se_b0 = s * np.sqrt(1 / n + x_bar**2 / Sxx)

ci_b0 = (b0 - t_crit * se_b0, b0 + t_crit * se_b0)
ci_b1 = (b1 - t_crit * se_b1, b1 + t_crit * se_b1)

print(f"se(b0) = {se_b0:.4f}   se(b1) = {se_b1:.5f}   t(0,025; {n - 2}) = {t_crit:.4f}")
print(f"CI 95% cho beta0: ({ci_b0[0]:.3f}; {ci_b0[1]:.3f})")
print(f"CI 95% cho beta1: ({ci_b1[0]:.4f}; {ci_b1[1]:.4f})")

# Đối chiếu với statsmodels
print("Khớp statsmodels:", np.allclose(res_check.conf_int(0.05), [ci_b0, ci_b1]))

#### 3.1.2 Khoảng tin cậy cho các hệ số của `Abdomen`

Khoảng tin cậy đối xứng 95% cho $\beta_1$ và $\beta_0$ có dạng

$$\hat\beta_1 \pm t_{0{,}025;250}\,\frac{s}{\sqrt{S_{xx}}},\qquad \hat\beta_0 \pm t_{0{,}025;250}\; s\sqrt{\frac{1}{n}+\frac{\bar x^2}{S_{xx}}}.$$

- **Hệ số góc $\beta_1$:** $0{,}6313 \pm 1{,}9695 \times 0{,}02855$, tức $(0{,}5751;\ 0{,}6875)$. Khoảng này không chứa 0 nên phù hợp với việc bác bỏ $H_0: \beta_1 = 0$. Như vậy, mỗi cm vòng bụng tăng thêm làm `BodyFat` trung bình tăng **ít nhất khoảng 0,58 và nhiều nhất khoảng 0,69** điểm phần trăm.
- **Hệ số chặn $\beta_0$:** $-39{,}28 \pm 1{,}9695 \times 2{,}6603$, tức $(-44{,}52;\ -34{,}04)$. Vì $x = 0$ không có nghĩa nên con số này chỉ nêu cho đủ.

In [ ]:
# --- Hồi quy đơn bằng statsmodels cho cả 4 biến (Abdomen để đối chiếu với phần tính tay) ---
ket_qua = {}
for x_ten in xs:
    X = sm.add_constant(df[x_ten])
    ket_qua[x_ten] = sm.OLS(df["BodyFat"], X).fit()

def tom_tat(res, x_ten):
    ci = res.conf_int(0.05).loc[x_ten]
    return pd.Series({
        "b0": res.params["const"], "b1": res.params[x_ten], "se(b1)": res.bse[x_ten],
        "t0": res.tvalues[x_ten], "p-value": res.pvalues[x_ten],
        "CI95 thấp": ci.iloc[0], "CI95 cao": ci.iloc[1],
        "s": np.sqrt(res.mse_resid), "R²": res.rsquared,
    })

bang = pd.DataFrame({x_ten: tom_tat(ket_qua[x_ten], x_ten) for x_ten in xs}).T

t_crit = stats.t.ppf(0.975, len(df) - 2)
print(f"Miền bác bỏ W = (-inf; {-t_crit:.4f}) U ({t_crit:.4f}; +inf)")

hien_thi = bang.drop(columns="p-value").apply(lambda c: c.map("{:.4f}".format))   # 4 chữ số thập phân
hien_thi.insert(4, "t0 thuộc W?", np.where(bang["t0"].abs() > t_crit, "Có", "Không"))
hien_thi.insert(5, "p-value", bang["p-value"].map(lambda v: f"{v:.2e}"))           # p-value (chỉ để đối chiếu)
display(hien_thi)

# Khoảng tin cậy 95% cho beta0 của cả 4 biến (beta1 đã có trong bảng trên)
ci_b0 = pd.DataFrame({x_ten: ket_qua[x_ten].conf_int(0.05).loc["const"].to_numpy() for x_ten in xs},
                     index=["CI95 thấp", "CI95 cao"]).T
print("Khoảng tin cậy 95% cho beta0:")
display(ci_b0.round(3))

### 3.2 `Wrist`

**Phương trình:** $\widehat{\text{BodyFat}} = -37{,}48 + 3{,}107\,\text{Wrist}$. Cổ tay lớn hơn 1 cm thì `BodyFat` trung bình cao hơn khoảng 3,1 điểm phần trăm. Hệ số góc lớn vì vòng cổ tay biến thiên rất hẹp (độ lệch chuẩn khoảng 0,93 cm), nên 1 cm đã là chênh lệch lớn giữa hai người; không nên so sánh trực tiếp độ lớn $\hat\beta_1$ giữa các biến có đơn vị khác nhau.

**Kiểm định ý nghĩa của hồi quy** ($\alpha = 5\%$):

1. $H_0: \beta_1 = 0$, $H_1: \beta_1 \ne 0$.
2. $T_0 = \hat\beta_1/(S/\sqrt{S_{xx}}) \sim t(250)$ nếu $H_0$ đúng.
3. $t_0 = 5{,}84$.
4. $W_\alpha = (-\infty;\,-1{,}9695) \cup (1{,}9695;\,+\infty)$.
5. $t_0 = 5{,}84 \in W_\alpha$ nên **bác bỏ $H_0$**. (Bằng ANOVA: $f_0 = t_0^2 = 34{,}13 \in (3{,}8789;\,+\infty)$, cùng kết luận.)

**Khoảng tin cậy 95% cho $\beta_1$:** $(2{,}059;\ 4{,}154)$, không chứa 0. Khoảng tin cậy 95% cho $\beta_0$ là $(-56{,}60;\ -18{,}37)$.

**Hệ số xác định:** $R^2 = 0{,}1201$: vòng cổ tay chỉ giải thích được khoảng **12,0%** biến thiên của `BodyFat`; độ lệch chuẩn sai số $s \approx 7{,}87$ còn lớn, nên dự đoán chỉ từ cổ tay kém chính xác.

### 3.3 `Age`

**Phương trình:** $\widehat{\text{BodyFat}} = 10{,}46 + 0{,}1936\,\text{Age}$. Mỗi năm tuổi tăng thêm thì `BodyFat` trung bình tăng khoảng 0,19 điểm phần trăm (khoảng 1,9 điểm phần trăm cho mỗi 10 năm).

**Kiểm định ý nghĩa của hồi quy** ($\alpha = 5\%$):

1. $H_0: \beta_1 = 0$, $H_1: \beta_1 \ne 0$.
2. $T_0 = \hat\beta_1/(S/\sqrt{S_{xx}}) \sim t(250)$ nếu $H_0$ đúng.
3. $t_0 = 4{,}82$.
4. $W_\alpha = (-\infty;\,-1{,}9695) \cup (1{,}9695;\,+\infty)$.
5. $t_0 = 4{,}82 \in W_\alpha$ nên **bác bỏ $H_0$**. (Bằng ANOVA: $f_0 = t_0^2 = 23{,}21 \in (3{,}8789;\,+\infty)$, cùng kết luận.)

**Khoảng tin cậy 95% cho $\beta_1$:** $(0{,}1144;\ 0{,}2727)$, không chứa 0. Khoảng tin cậy 95% cho $\beta_0$ là $(6{,}77;\ 14{,}15)$.

**Hệ số xác định:** $R^2 = 0{,}0849$: tuổi giải thích được khoảng **8,5%** biến thiên của `BodyFat`.

### 3.4 `Height`

**Phương trình:** $\widehat{\text{BodyFat}} = 24{,}85 - 0{,}0811\,\text{Height}$. Hệ số góc âm rất nhỏ (cao hơn 1 inch thì `BodyFat` thấp hơn khoảng 0,08 điểm phần trăm); kiểm định dưới đây cho thấy hệ số này không khác 0 có ý nghĩa.

**Kiểm định ý nghĩa của hồi quy** ($\alpha = 5\%$):

1. $H_0: \beta_1 = 0$, $H_1: \beta_1 \ne 0$.
2. $T_0 = \hat\beta_1/(S/\sqrt{S_{xx}}) \sim t(250)$ nếu $H_0$ đúng.
3. $t_0 = -0{,}40$.
4. $W_\alpha = (-\infty;\,-1{,}9695) \cup (1{,}9695;\,+\infty)$.
5. $t_0 = -0{,}40 \notin W_\alpha$ nên **chưa có cơ sở để bác bỏ $H_0$**: với dữ liệu này chưa đủ bằng chứng về quan hệ tuyến tính giữa chiều cao và `BodyFat`. (Điều này không có nghĩa là đã chứng minh được $\beta_1 = 0$.) Bằng ANOVA: $f_0 = t_0^2 = 0{,}16 \notin (3{,}8789;\,+\infty)$, cùng kết luận.

**Khoảng tin cậy 95% cho $\beta_1$:** $(-0{,}4804;\ 0{,}3182)$, **chứa 0**, phù hợp với kết luận chưa bác bỏ được $H_0$. Khoảng tin cậy 95% cho $\beta_0$ là $(-3{,}24;\ 52{,}95)$.

**Hệ số xác định:** $R^2 = 0{,}0006$: chiều cao giải thích được khoảng **0,06%** biến thiên của `BodyFat`, tức gần như không đáng kể.

**Nhận xét chung.** Ở mức ý nghĩa 5%, ba biến `Abdomen`, `Wrist`, `Age` có $\beta_1$ khác 0 có ý nghĩa thống kê, riêng `Height` thì không. Tuy nhiên "có ý nghĩa thống kê" không đồng nghĩa với "dự đoán tốt": chỉ `Abdomen` có $R^2$ lớn, còn `Wrist` và `Age` giải thích được rất ít biến thiên của `BodyFat`. Các kết luận trên dựa trên giả thiết sai số độc lập, có phương sai không đổi và xấp xỉ chuẩn của mô hình.

## 4. So sánh và kết luận

Bảng dưới tổng hợp kết quả hồi quy đơn của cả bốn biến (sắp xếp theo $R^2$ giảm dần); miền bác bỏ ở mức ý nghĩa 5% là $W_\alpha = (-\infty;\,-1{,}9695) \cup (1{,}9695;\,+\infty)$.

In [ ]:
t_crit = stats.t.ppf(0.975, len(df) - 2)

tong_hop = pd.DataFrame({
    "b1": bang["b1"],
    "se(b1)": bang["se(b1)"],
    "t0": bang["t0"],
    "t0 thuộc W?": np.where(bang["t0"].abs() > t_crit, "Có", "Không"),
    "CI95 thấp": bang["CI95 thấp"],
    "CI95 cao": bang["CI95 cao"],
    "R²": bang["R²"],
    "R² (%)": 100 * bang["R²"],
}).sort_values("R²", ascending=False)
hien_thi = tong_hop.copy()
for cot_ten in ["b1", "se(b1)", "t0", "CI95 thấp", "CI95 cao", "R²"]:
    hien_thi[cot_ten] = tong_hop[cot_ten].map("{:.4f}".format)      # 4 chữ số thập phân, tránh dạng khoa học
hien_thi["R² (%)"] = tong_hop["R² (%)"].map("{:.2f}".format)
display(hien_thi)

# Biểu đồ so sánh R² giữa các biến
fig, ax = plt.subplots(figsize=(6, 3.8))
cot = ax.bar(tong_hop.index, tong_hop["R² (%)"])
ax.bar_label(cot, fmt="%.2f%%")
ax.set_ylim(0, tong_hop["R² (%)"].max() * 1.15)      # chừa chỗ cho nhãn
ax.set_ylabel("R² (%)")
ax.set_title("Phần trăm biến thiên của BodyFat được mỗi biến x giải thích")
plt.tight_layout()
plt.show()

**So sánh $R^2$.** `Abdomen` có $R^2$ cao nhất ($66{,}2\%$), gấp hơn 5 lần `Wrist` ($12{,}0\%$); tiếp theo là `Age` ($8{,}5\%$) và cuối cùng là `Height` ($0{,}06\%$, gần như bằng 0).

**Ý nghĩa thống kê của hệ số góc.** Ở mức ý nghĩa 5%, `Abdomen`, `Wrist` và `Age` đều có $t_0$ thuộc miền bác bỏ, nên bác bỏ $H_0: \beta_1 = 0$: cả ba biến đều có ảnh hưởng tuyến tính có ý nghĩa thống kê đến `BodyFat`, và ảnh hưởng đều **cùng chiều thuận** (khoảng tin cậy 95% của $\beta_1$ nằm hoàn toàn phía dương). Riêng `Height` có $t_0 = -0{,}40$ không thuộc miền bác bỏ và khoảng tin cậy $(-0{,}4804;\ 0{,}3182)$ chứa 0, nên chưa có cơ sở để bác bỏ $H_0$: với dữ liệu này chưa thấy bằng chứng về quan hệ tuyến tính giữa chiều cao và `BodyFat`, đúng với vai trò biến đối chứng.

**"Có ý nghĩa thống kê" không đồng nghĩa với "giải thích tốt".** `Wrist` và `Age` có $\beta_1$ khác 0 một cách có ý nghĩa, nhưng mỗi biến chỉ giải thích được khoảng 12% và 8,5% biến thiên của `BodyFat`; phần lớn biến thiên còn lại do các yếu tố khác không có trong mô hình đó. Chỉ `Abdomen` giải thích được phần lớn (hơn hai phần ba) biến thiên. Cũng lưu ý độ lớn của $\hat\beta_1$ không so sánh trực tiếp được giữa các biến vì khác đơn vị (cm, năm, inch), việc so sánh giữa các biến dựa vào $R^2$.

**Kết luận chính.** Vòng bụng là số đo giải thích phần trăm mỡ cơ thể tốt nhất trong bốn biến, với phương trình $\widehat{\text{BodyFat}} = -39{,}28 + 0{,}6313\,\text{Abdomen}$: vòng bụng tăng thêm 1 cm thì phần trăm mỡ trung bình tăng thêm khoảng 0,63 điểm phần trăm (khoảng tin cậy 95%: từ 0,58 đến 0,69). Vòng cổ tay và tuổi chỉ có liên hệ yếu, còn chiều cao gần như không có liên hệ tuyến tính với phần trăm mỡ.

**Hạn chế của báo cáo.**

- Mỗi mô hình chỉ có một biến $x$ nên phần biến thiên còn lại của `BodyFat` (đặc biệt với `Wrist`, `Age`) do các yếu tố chưa đưa vào mô hình.
- Dữ liệu chỉ gồm nam giới (252 người), nên kết quả không tự suy rộng cho nữ giới hay các nhóm khác.
- Các kết quả kiểm định và khoảng tin cậy dựa trên giả thiết sai số độc lập, có phương sai không đổi và xấp xỉ chuẩn; báo cáo không kiểm tra lại các giả thiết này.
- Một số điểm dữ liệu đáng ngờ (case 39, 182, 48, 76) được giữ nguyên; đặc biệt case 39 có vòng bụng 148,1 cm rất xa so với phần còn lại của mẫu nên có thể ảnh hưởng đến kết quả của `Abdomen`.
- Hệ số chặn $\hat\beta_0$ ứng với $x = 0$ không có ý nghĩa thực tế nên không được diễn giải.

**Tài liệu tham khảo.**

- Johnson, R. W. (1996). Fitting percentage of body fat to simple body measurements. *Journal of Statistics Education*, 4(1). <https://jse.amstat.org/datasets/fat.txt>
- Nguyễn Thị Thu Thủy, bài giảng MI3031 – Chương 4, mục 4.1: Hồi quy tuyến tính đơn (HUST, 2025).